# Urban Safety — Regression (Crime Count): Feature Inspection & Pre-ML Analysis

Loads pre-computed segment features, explores the crime count target variable and feature distributions prior to regression modelling.

**See `03_London_RegressionML_CrimeCount.ipynb` for model training.**

## Cell 1 — Imports and config

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler

sns.set(rc={'figure.figsize': (8, 8)})
pd.set_option('display.float_format', '{:.3f}'.format)

FEATURE_COLS = ['lighting', 'visibility', 'connectivity', 'enclosure']

## Cell 2 — Load pre-computed features

In [ ]:
CRIME_FEATURES_CSV = 'csvFiles/segment_crime_scores_w-id.csv'
features = pd.read_csv(CRIME_FEATURES_CSV)

required_cols = FEATURE_COLS + ['crime_count', 'location_id', 'borough']
missing = [c for c in required_cols if c not in features.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

print(f'\u2713 Loaded {len(features):,} segments with {len(features.columns)} columns')
display(features.head())
print(f'\nCrime count statistics:')
print(features['crime_count'].describe().round(3))
print(f'\nFeature statistics:')
display(features[FEATURE_COLS].describe().round(3))

## Cell 3 — Crime Count Distribution Check
Inspect the distribution of total crime counts. This is the regression target variable.

In [ ]:
print('=== CRIME COUNT DISTRIBUTION ===\n')
crime_counts = features['crime_count'].values
print(f'Total segments: {len(crime_counts):,}')
for label, val in [('Min', crime_counts.min()), ('Q1', np.percentile(crime_counts, 25)),
                   ('Median', np.percentile(crime_counts, 50)), ('Mean', crime_counts.mean()),
                   ('Q3', np.percentile(crime_counts, 75)), ('Max', crime_counts.max()),
                   ('Std Dev', crime_counts.std())]:
    print(f'  {label:8s}: {val:.2f}')
zero = (crime_counts == 0).sum()
print(f'\n  Zero crime: {zero:,} ({100*zero/len(crime_counts):.1f}%)  '
      f'Non-zero: {len(crime_counts)-zero:,} ({100*(len(crime_counts)-zero)/len(crime_counts):.1f}%)')

print(f'\nPer-borough mean crime count:')
bm = features.groupby('borough')['crime_count'].agg(['mean','count']).sort_values('mean', ascending=False)
bm.index = [b.replace('London Borough of ','').replace(', London, UK','').replace('City of ','') for b in bm.index]
print(bm.round(2).to_string())

In [ ]:
os.makedirs('plots', exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(features['crime_count'], bins=80, color='steelblue', alpha=0.7, edgecolor='none')
axes[0].set_xlabel('Total Crime Count'); axes[0].set_ylabel('Number of Segments')
axes[0].set_title('Distribution of Crime Counts Across All Segments')
axes[0].spines[['top','right']].set_visible(False); axes[0].grid(True, alpha=0.3, axis='y', linestyle='--')

nonzero = features[features['crime_count'] > 0]['crime_count']
axes[1].hist(nonzero, bins=80, color='coral', alpha=0.7, edgecolor='none')
axes[1].set_xlabel('Total Crime Count'); axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_yscale('log')
axes[1].set_title(f'Distribution of Crime Counts (Non-Zero Only, n={len(nonzero):,})')
axes[1].spines[['top','right']].set_visible(False); axes[1].grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plt.savefig('plots/crime_count_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
fc = features.copy()
fc['borough_short'] = (fc['borough'].str.replace('London Borough of ','')
    .str.replace(', London, UK','').str.replace('City of ',''))
fc.boxplot(column='crime_count', by='borough_short', ax=ax)
ax.set_xlabel('Borough'); ax.set_ylabel('Crime Count')
ax.set_title('Crime Count Distribution by Borough')
plt.suptitle(''); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/crime_count_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature EDA — inspect the data and how it correlates

In [ ]:
feat_df = features[FEATURE_COLS].copy()
print(f'Feature dataframe: {feat_df.shape[0]:,} segments x {feat_df.shape[1]} features')
display(feat_df.head(10))
summary = feat_df.describe().T
summary['skew'] = feat_df.skew()
print('Summary (with skew):')
display(summary.round(3))

import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style='whitegrid')
plt.figure(figsize=(6, 5))
corr = features[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature correlation matrix (Pearson)')
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr.round(3))

In [ ]:
from pandas.plotting import parallel_coordinates
from matplotlib.patches import Patch

_z = (features[FEATURE_COLS] - features[FEATURE_COLS].mean()) / features[FEATURE_COLS].std()
_z['crime_count'] = features['crime_count'].values
_z = _z.dropna(subset=['crime_count']).sample(n=min(800, len(_z)), random_state=42)

all_nonzero = features[features['crime_count'] > 0]['crime_count']
q25 = all_nonzero.quantile(0.25); q50 = all_nonzero.quantile(0.50)
q75 = all_nonzero.quantile(0.75); max_v = features['crime_count'].max()

_color_map = {'Very Low': '#808080', 'Low': '#2ca02c', 'Medium': '#ffd700',
               'High': '#ff7f0e', 'Very High': '#d62728'}

fig, ax = plt.subplots(figsize=(11, 6))
_z['crime_count_bin'] = pd.cut(_z['crime_count'],
    bins=[-0.1, 0.0001, q25, q50, q75, max_v + 1],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'], include_lowest=True)
_present = sorted(_z['crime_count_bin'].unique().tolist())
parallel_coordinates(_z[FEATURE_COLS + ['crime_count_bin']], 'crime_count_bin',
                     color=[_color_map[c] for c in _present], alpha=0.5, ax=ax)
ax.legend(handles=[Patch(facecolor=_color_map[c], label=c) for c in _present])
ax.set_title('Parallel coordinates by crime count (z-scored features, sampled)')
ax.set_ylabel('Feature value (z-scored)')
_y_min = min(_z[FEATURE_COLS].min().min(), -3); _y_max = max(_z[FEATURE_COLS].max().max(), 3)
ax.set_ylim(_y_min - 0.3, _y_max + 0.3)
plt.tight_layout()
plt.savefig('plots/parallel_coordinates.png', dpi=150, bbox_inches='tight')
plt.show()

# Excluding zero-score segments
_znz = (features[FEATURE_COLS] - features[FEATURE_COLS].mean()) / features[FEATURE_COLS].std()
_znz['crime_count'] = features['crime_count'].values
_znz = _znz[_znz['crime_count'] > 0].sample(n=min(800, len(_znz)), random_state=42)
_znz['crime_count_bin'] = pd.cut(_znz['crime_count'],
    bins=[-0.1, 0.0001, q25, q50, q75, max_v + 1],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'], include_lowest=True)
_znz_f = _znz[_znz['crime_count_bin'] != 'Very Low']
_present2 = sorted(_znz_f['crime_count_bin'].unique().tolist())
fig, ax = plt.subplots(figsize=(11, 6))
parallel_coordinates(_znz_f[FEATURE_COLS + ['crime_count_bin']], 'crime_count_bin',
                     color=[_color_map[c] for c in _present2], alpha=0.5, ax=ax)
ax.legend(handles=[Patch(facecolor=_color_map[c], label=c) for c in _present2])
ax.set_title('Parallel coordinates by crime count - excluding zero-score segments (sampled)')
ax.set_ylabel('Feature value (z-scored)')
ax.set_ylim(_y_min - 0.3, _y_max + 0.3)
plt.tight_layout()
plt.savefig('plots/parallel_coordinates_nonzero.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Non-zero segments shown: {len(_znz_f):,}')

## Target Variable: Total Crime Count

In [ ]:
print('=== Crime Count Target Variable ===\n')
cc = features['crime_count']
print(f'Total segments: {len(features):,}')
print(f'Segments with crimes: {(cc > 0).sum():,}')
print(f'Segments with no crimes: {(cc == 0).sum():,}')
print(f'\nCrime count statistics:')
for label, val in [('Min', cc.min()), ('Max', cc.max()), ('Mean', cc.mean()),
                   ('Median', cc.median()), ('Std Dev', cc.std()),
                   ('Q1 (25%)', cc.quantile(0.25)), ('Q3 (75%)', cc.quantile(0.75)),
                   ('IQR', cc.quantile(0.75) - cc.quantile(0.25))]:
    print(f'  {label:12s}: {val:.2f}')
print(f'\nCrime count percentiles:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p:2d}: {cc.quantile(p/100):6.0f}')
print(f'\nTotal crimes across all segments: {cc.sum():,.0f}')

## Standard scale the crime count

In [ ]:
crime_counts = features['crime_count'].values
scaler_cc = StandardScaler()
scaled_crime_counts = scaler_cc.fit_transform(crime_counts.reshape(-1, 1)).flatten()

print(f'Original crime counts  - Min: {crime_counts.min():.2f}, Max: {crime_counts.max():.2f}, '
      f'Mean: {crime_counts.mean():.2f}, Std: {crime_counts.std():.2f}')
print(f'Scaled crime counts    - Min: {scaled_crime_counts.min():.2f}, Max: {scaled_crime_counts.max():.2f}, '
      f'Mean: {scaled_crime_counts.mean():.6f}, Std: {scaled_crime_counts.std():.2f}')
print(f'Total segments: {len(scaled_crime_counts)}')